# Adaptive Darcy macro meshes

The published sine problem of Barrenechea et al. (2026), DOI [10.1137/24M1673073](https://doi.org/10.1137/24M1673073), is evaluated using the weighted energy decomposition and an original Dörfler/red-green adaptive policy. Neither contraction nor monotonicity is assumed.

In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm import FaceSpace, SkeletonSpace, TriangleMesh, solve_darcy
from pymhm.cut_cells import fit_material_faces, fit_material_mesh
from pymhm.reservoir import CartesianCellField
from pymhm.adaptive_darcy import solve_adaptive_darcy
root = Path.cwd()

In [ ]:
macro = TriangleMesh.unit_square()
sine = lambda x: np.prod(np.sin(np.pi*x), axis=1)
with threadpool_limits(1):
    adaptive = solve_adaptive_darcy(macro, iterations=2, theta=0.5, local_refinement=2,
        source=lambda x: 2*np.pi**2*sine(x), estimator_order=8)
print({"macro_cells": [len(s.skeleton.mesh.cells) for s in adaptive.solutions],
       "estimators": adaptive.totals.tolist()})

## Five-level campaign

The archived campaign starts with eight macrotriangles and uses $p=\sin(2\pi x)\sin(2\pi y)$. Both $\ell=0,k=2$ and $\ell=1,k=3$ use RT2 reconstruction, four local triangles per macro, and $\theta=0.5$. The P0 trace sequence has a nonmonotone error history; all ten measured estimators exceed their independently integrated energy errors. Marked cells are blue and all field panels show the actual macro geometry.

In [ ]:
records = json.loads((root / "examples/results/adaptive-darcy/campaign.json").read_text())
for row in records:
    print(row["trace_degree"], row["level"], row["macro_triangles"], row["energy_error"], row["estimator"])
for name in ("refinement", "meshes-ell1", "fields"):
    display(Image(filename=str(root / "docs/figures/adaptive-darcy" / f"{name}.png")))